In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import math

# Создаём Spark Session
spark = SparkSession.builder \
    .appName("L1_Bike_Analysis") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark Version:", spark.version)
print("Spark UI: http://localhost:4040")

Spark Version: 3.3.0
Spark UI: http://localhost:4040


In [2]:
# Загружаем данные из CSV файлов
tripData = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("timestampFormat", "M/d/y H:m") \
    .csv("trip.csv")

stationData = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("timestampFormat", "M/d/y") \
    .csv("station.csv")

# Очищаем данные от NULL значений в критических полях
tripData = tripData.dropna(subset=["duration", "start_station_id", "end_station_id", "bike_id"])
stationData = stationData.dropna(subset=["id", "lat", "long"])

print("=== Схема данных о поездках ===")
tripData.printSchema()

print("\n=== Схема данных о станциях ===")
stationData.printSchema()

print(f"\nВсего поездок: {tripData.count()}")
print(f"Всего станций: {stationData.count()}")

=== Схема данных о поездках ===
root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)


=== Схема данных о станциях ===
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: timestamp (nullable = true)


Всего поездок: 669959
Всего станций: 70


In [3]:
# Показываем примеры данных
print("=== Пример данных о поездках ===")
tripData.show(5, truncate=False)

print("\n=== Пример данных о станциях ===")
stationData.show(5, truncate=False)

=== Пример данных о поездках ===
+----+--------+-------------------+------------------------+----------------+-------------------+------------------------+--------------+-------+-----------------+--------+
|id  |duration|start_date         |start_station_name      |start_station_id|end_date           |end_station_name        |end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+------------------------+----------------+-------------------+------------------------+--------------+-------+-----------------+--------+
|4576|63      |2013-08-29 14:13:00|South Van Ness at Market|66              |2013-08-29 14:14:00|South Van Ness at Market|66            |520    |Subscriber       |94127   |
|4607|70      |2013-08-29 14:42:00|San Jose City Hall      |10              |2013-08-29 14:43:00|San Jose City Hall      |10            |661    |Subscriber       |95138   |
|4130|71      |2013-08-29 10:16:00|Mountain View City Hall |27              |2013-08-29 10:17:00|Mount

In [4]:
# ============================================================================
# ЗАДАНИЕ 1: Найти велосипед с максимальным временем пробега
# ============================================================================

# Группируем по bike_id и суммируем duration (в секундах)
bikeTotalDuration = tripData.groupBy("bike_id") \
    .agg(F.sum("duration").alias("total_duration_sec"),
         F.count("id").alias("trip_count")) \
    .orderBy(F.col("total_duration_sec").desc())

# Находим велосипед с максимальным временем
maxBike = bikeTotalDuration.first()

print("=" * 70)
print("ЗАДАНИЕ 1: Велосипед с максимальным временем пробега")
print("=" * 70)
print(f"ID велосипеда: {maxBike['bike_id']}")
print(f"Общее время в пути: {maxBike['total_duration_sec']} секунд")
print(f"Общее время в пути: {maxBike['total_duration_sec'] / 3600:.2f} часов")
print(f"Количество поездок: {maxBike['trip_count']}")
print("=" * 70)

# Показываем топ-10 велосипедов
print("\nТоп-10 велосипедов по общему времени в пути:")
bikeTotalDuration.show(10, truncate=False)

ЗАДАНИЕ 1: Велосипед с максимальным временем пробега
ID велосипеда: 535
Общее время в пути: 18611693 секунд
Общее время в пути: 5169.91 часов
Количество поездок: 1328

Топ-10 велосипедов по общему времени в пути:
+-------+------------------+----------+
|bike_id|total_duration_sec|trip_count|
+-------+------------------+----------+
|535    |18611693          |1328      |
|466    |3933272           |1507      |
|613    |2409014           |1867      |
|526    |2253019           |1838      |
|415    |2248886           |1869      |
|572    |2234149           |1718      |
|524    |2214314           |1915      |
|542    |2213422           |1768      |
|465    |2185170           |1813      |
|376    |2178177           |1787      |
+-------+------------------+----------+
only showing top 10 rows



In [5]:
# ============================================================================
# ЗАДАНИЕ 2: Найти наибольшее геодезическое расстояние между станциями
# ============================================================================

# Функция для расчёта расстояния между двумя точками (формула гаверсинуса)
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Вычисляет расстояние между двумя точками на сфере Земли в километрах
    """
    R = 6371  # Радиус Земли в км
    
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    delta_lat = math.radians(lat2 - lat1)
    delta_lon = math.radians(lon2 - lon1)
    
    a = math.sin(delta_lat / 2) ** 2 + \
        math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(delta_lon / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    return R * c

# Регистрируем UDF для использования в Spark SQL
haversine_udf = F.udf(haversine_distance, DoubleType())

# Создаём декартово произведение станций для вычисления всех пар расстояний
# (для оптимизации берём только уникальные пары где id1 < id2)
stations_with_coords = stationData.select(
    F.col("id").alias("station1_id"),
    F.col("name").alias("station1_name"),
    F.col("lat").alias("lat1"),
    F.col("long").alias("lon1")
)

stations_cross = stations_with_coords.crossJoin(
    stationData.select(
        F.col("id").alias("station2_id"),
        F.col("name").alias("station2_name"),
        F.col("lat").alias("lat2"),
        F.col("long").alias("lon2")
    )
)

# Фильтруем чтобы не было дубликатов и одинаковых станций
station_pairs = stations_cross.filter(
    (F.col("station1_id") < F.col("station2_id"))
)

# Вычисляем расстояние для каждой пары
station_distances = station_pairs.withColumn(
    "distance_km",
    haversine_udf(F.col("lat1"), F.col("lon1"), F.col("lat2"), F.col("lon2"))
)

# Находим максимальное расстояние
max_distance = station_distances.orderBy(F.col("distance_km").desc()).first()

print("=" * 70)
print("ЗАДАНИЕ 2: Наибольшее геодезическое расстояние между станциями")
print("=" * 70)
print(f"Станция 1: {max_distance['station1_name']} (ID: {max_distance['station1_id']})")
print(f"Станция 2: {max_distance['station2_name']} (ID: {max_distance['station2_id']})")
print(f"Расстояние: {max_distance['distance_km']:.2f} км")
print(f"Координаты станции 1: {max_distance['lat1']}, {max_distance['lon1']}")
print(f"Координаты станции 2: {max_distance['lat2']}, {max_distance['lon2']}")
print("=" * 70)

# Показываем топ-10 самых удалённых пар станций
print("\nТоп-10 самых удалённых пар станций:")
station_distances.orderBy(F.col("distance_km").desc()).show(10, truncate=False)

ЗАДАНИЕ 2: Наибольшее геодезическое расстояние между станциями
Станция 1: SJSU - San Salvador at 9th (ID: 16)
Станция 2: Embarcadero at Sansome (ID: 60)
Расстояние: 69.92 км
Координаты станции 1: 37.333954999999996, -121.877349
Координаты станции 2: 37.80477, -122.40323400000001

Топ-10 самых удалённых пар станций:
+-----------+--------------------------+------------------+-------------------+-----------+-------------------------------+---------+-------------------+-----------------+
|station1_id|station1_name             |lat1              |lon1               |station2_id|station2_name                  |lat2     |lon2               |distance_km      |
+-----------+--------------------------+------------------+-------------------+-----------+-------------------------------+---------+-------------------+-----------------+
|16         |SJSU - San Salvador at 9th|37.333954999999996|-121.877349        |60         |Embarcadero at Sansome         |37.80477 |-122.40323400000001|69.92087595428

In [6]:
# ============================================================================
# ЗАДАНИЕ 3: Найти путь велосипеда с максимальным временем пробега через станции
# ============================================================================

# ID велосипеда с максимальным временем (из задания 1)
max_bike_id = maxBike['bike_id']

# Получаем все поездки этого велосипеда, отсортированные по времени начала
bike_trips = tripData.filter(F.col("bike_id") == max_bike_id) \
    .orderBy(F.col("start_date")) \
    .select(
        "id",
        "start_date",
        "end_date",
        "duration",
        "start_station_id",
        "start_station_name",
        "end_station_id",
        "end_station_name"
    )

# Присоединяем информацию о станциях для получения координат
bike_trips_with_stations = bike_trips.join(
    stationData.select(F.col("id").alias("start_station_id_ref"), 
                       F.col("lat").alias("start_lat"),
                       F.col("long").alias("start_lon")),
    bike_trips.start_station_id == F.col("start_station_id_ref"),
    "left"
).join(
    stationData.select(F.col("id").alias("end_station_id_ref"),
                       F.col("lat").alias("end_lat"),
                       F.col("long").alias("end_lon")),
    bike_trips.end_station_id == F.col("end_station_id_ref"),
    "left"
).drop("start_station_id_ref", "end_station_id_ref")

# Собираем все поездки в список для отображения пути
bike_path = bike_trips_with_stations.collect()

print("=" * 70)
print(f"ЗАДАНИЕ 3: Путь велосипеда ID {max_bike_id} через станции")
print("=" * 70)
print(f"Всего поездок велосипеда: {len(bike_path)}")
print(f"Общее время в пути: {maxBike['total_duration_sec']} секунд ({maxBike['total_duration_sec'] / 3600:.2f} часов)")
print("=" * 70)

print("\nПоследовательность поездок (первые 20):")
print("-" * 70)
for i, trip in enumerate(bike_path[:20], 1):
    print(f"{i:2d}. {trip['start_date']} | {trip['start_station_name'][:30]:30s} → {trip['end_station_name'][:30]:30s} | {trip['duration']:4d} сек")

print("-" * 70)
if len(bike_path) > 20:
    print(f"... и ещё {len(bike_path) - 20} поездок")

# Показываем все поездки в виде таблицы
print("\nВсе поездки велосипеда:")
bike_trips_with_stations.show(truncate=False)

ЗАДАНИЕ 3: Путь велосипеда ID 535 через станции
Всего поездок велосипеда: 1328
Общее время в пути: 18611693 секунд (5169.91 часов)

Последовательность поездок (первые 20):
----------------------------------------------------------------------
 1. 2013-08-29 21:38:00 | San Francisco Caltrain (Townse → San Francisco Caltrain 2 (330  |  423 сек
 2. 2013-08-29 19:32:00 | Post at Kearney                → San Francisco Caltrain (Townse | 1245 сек
 3. 2013-08-30 09:10:00 | Market at Sansome              → 2nd at South Park              |  498 сек
 4. 2013-08-30 08:40:00 | San Francisco Caltrain 2 (330  → Market at Sansome              |  842 сек
 5. 2013-09-01 12:58:00 | 2nd at Townsend                → Davis at Jackson               | 1671 сек
 6. 2013-09-05 11:59:00 | San Francisco City Hall        → Civic Center BART (7th at Mark |  260 сек
 7. 2013-09-06 10:55:00 | Civic Center BART (7th at Mark → Post at Kearney                | 1192 сек
 8. 2013-09-06 13:58:00 | Post at Kearney         

In [7]:
# ============================================================================
# ЗАДАНИЕ 4: Найти количество велосипедов в системе
# ============================================================================

# Подсчитываем уникальные bike_id в данных о поездках
unique_bikes = tripData.select("bike_id").distinct()
bike_count = unique_bikes.count()

# Статистика по велосипедам
bike_stats = tripData.groupBy("bike_id").agg(
    F.count("id").alias("trip_count"),
    F.sum("duration").alias("total_duration"),
    F.avg("duration").alias("avg_duration")
)

print("=" * 70)
print("ЗАДАНИЕ 4: Количество велосипедов в системе")
print("=" * 70)
print(f"Общее количество уникальных велосипедов: {bike_count}")
print("=" * 70)

# Дополнительная статистика
print("\nСтатистика использования велосипедов:")
bike_stats_summary = bike_stats.agg(
    F.avg("trip_count").alias("avg_trips_per_bike"),
    F.max("trip_count").alias("max_trips_single_bike"),
    F.min("trip_count").alias("min_trips_single_bike"),
    F.avg("total_duration").alias("avg_total_duration_per_bike")
)
bike_stats_summary.show()

# Топ-5 самых используемых велосипедов
print("\nТоп-5 самых используемых велосипедов:")
bike_stats.orderBy(F.col("trip_count").desc()).show(5, truncate=False)

# Топ-5 наименее используемых велосипедов
print("\nТоп-5 наименее используемых велосипедов:")
bike_stats.orderBy(F.col("trip_count").asc()).show(5, truncate=False)

ЗАДАНИЕ 4: Количество велосипедов в системе
Общее количество уникальных велосипедов: 700

Статистика использования велосипедов:
+------------------+---------------------+---------------------+---------------------------+
|avg_trips_per_bike|max_trips_single_bike|min_trips_single_bike|avg_total_duration_per_bike|
+------------------+---------------------+---------------------+---------------------------+
| 957.0842857142857|                 2061|                    6|          1060401.387142857|
+------------------+---------------------+---------------------+---------------------------+


Топ-5 самых используемых велосипедов:
+-------+----------+--------------+-----------------+
|bike_id|trip_count|total_duration|avg_duration     |
+-------+----------+--------------+-----------------+
|392    |2061      |1789476       |868.2561863173216|
|489    |1975      |1745818       |883.9584810126582|
|558    |1955      |1729341       |884.5734015345269|
|267    |1951      |1791746       |918.3731

In [8]:
# ============================================================================
# ЗАДАНИЕ 5: Найти пользователей, потративших на поездки более 3 часов
# ============================================================================

# 3 часа = 3 * 3600 = 10800 секунд
THREE_HOURS_SECONDS = 10800

# Группируем по zip_code (идентификатор пользователя) и суммируем время
user_total_time = tripData.groupBy("zip_code") \
    .agg(
        F.sum("duration").alias("total_duration_sec"),
        F.count("id").alias("trip_count"),
        F.avg("duration").alias("avg_duration_sec")
    ) \
    .filter(F.col("total_duration_sec") > THREE_HOURS_SECONDS) \
    .orderBy(F.col("total_duration_sec").desc())

# Добавляем колонку с временем в часах для удобства
user_total_time_hours = user_total_time.withColumn(
    "total_duration_hours",
    F.round(F.col("total_duration_sec") / 3600, 2)
)

users_over_3h = user_total_time_hours.collect()

print("=" * 70)
print("ЗАДАНИЕ 5: Пользователи, потратившие на поездки более 3 часов")
print("=" * 70)
print(f"Порог: {THREE_HOURS_SECONDS} секунд (3 часа)")
print(f"Количество пользователей, превысивших порог: {len(users_over_3h)}")
print("=" * 70)

if len(users_over_3h) > 0:
    print("\nСписок пользователей (почтовые индексы):")
    print("-" * 70)
    print(f"{'№':<3} {'Zip Code':<12} {'Время (часы)':<15} {'Время (сек)':<12} {'Поездок':<10} {'Среднее (сек)':<12}")
    print("-" * 70)
    
    for i, user in enumerate(users_over_3h, 1):
        print(f"{i:<3} {str(user['zip_code']):<12} {user['total_duration_hours']:<15.2f} {user['total_duration_sec']:<12.0f} {user['trip_count']:<10} {user['avg_duration_sec']:<12.1f}")
    
    print("-" * 70)
    
    # Показываем топ-10 в виде DataFrame
    print("\nТоп-10 пользователей по общему времени в пути:")
    user_total_time_hours.show(10, truncate=False)
else:
    print("\nНет пользователей, потративших более 3 часов на поездки.")

# Дополнительный анализ по типам подписки
print("\n" + "=" * 70)
print("Дополнительно: Анализ по типу подписки")
print("=" * 70)

subscription_stats = tripData.groupBy("subscription_type").agg(
    F.count("id").alias("total_trips"),
    F.sum("duration").alias("total_duration_sec"),
    F.avg("duration").alias("avg_duration_sec"),
    F.countDistinct("zip_code").alias("unique_users")
)

subscription_stats.withColumn(
    "total_duration_hours",
    F.round(F.col("total_duration_sec") / 3600, 2)
).show(truncate=False)

ЗАДАНИЕ 5: Пользователи, потратившие на поездки более 3 часов
Порог: 10800 секунд (3 часа)
Количество пользователей, превысивших порог: 3661

Список пользователей (почтовые индексы):
----------------------------------------------------------------------
№   Zip Code     Время (часы)    Время (сек)  Поездок    Среднее (сек)
----------------------------------------------------------------------
1   94107        13821.43        49757162     78704      632.2       
2   nil          12701.54        45725550     10682      4280.6      
3   None         7700.91         27723273     6619       4188.4      
4   94105        7110.04         25596128     42672      599.8       
5   94133        6010.47         21637675     31359      690.0       
6   94102        5313.34         19128021     19757      968.2       
7   94103        5313.16         19127388     26673      717.1       
8   95531        4797.33         17270400     1          17270400.0  
9   94111        3956.94         14244997   

In [9]:
# ============================================================================
# СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ
# ============================================================================

print("\n" + "=" * 70)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ ЛАБОРАТОРНОЙ РАБОТЫ")
print("=" * 70)

results = {
    "Задание": [
        "1. Велосипед с макс. временем",
        "2. Макс. расстояние между станциями",
        "3. Путь велосипеда (кол-во поездок)",
        "4. Количество велосипедов",
        "5. Пользователей > 3 часов"
    ],
    "Результат": [
        f" Bike ID: {maxBike['bike_id']}",
        f" {max_distance['distance_km']:.2f} км",
        f" {len(bike_path)} поездок",
        f" {bike_count} велосипедов",
        f" {len(users_over_3h)} пользователей"
    ],
    "Дополнительно": [
        f" {maxBike['total_duration_sec']/3600:.2f} часов",
        f" {max_distance['station1_name'][:20]}... ↔ {max_distance['station2_name'][:20]}...",
        f" {maxBike['trip_count']} поездок этого байка",
        f" Среднее: {bike_stats_summary.first()['avg_trips_per_bike']:.1f} поездок/байк",
        f" Макс: {users_over_3h[0]['total_duration_hours']:.2f} часов" if users_over_3h else "N/A"
    ]
}

results_df = pd.DataFrame(results)
display(results_df)

print("=" * 70)


ИТОГОВЫЕ РЕЗУЛЬТАТЫ ЛАБОРАТОРНОЙ РАБОТЫ


,Задание,Результат,Дополнительно
0,1. Велосипед с макс. временем,Bike ID: 535,5169.91 часов
1,2. Макс. расстояние между станциями,69.92 км,SJSU - San Salvador ... ↔ Embarcadero at Sans...
2,3. Путь велосипеда (кол-во поездок),1328 поездок,1328 поездок этого байка
3,4. Количество велосипедов,700 велосипедов,Среднее: 957.1 поездок/байк
4,5. Пользователей > 3 часов,3661 пользователей,Макс: 13821.43 часов
